In [12]:
import requests
import requests, time

In [ ]:
# Extrair mercado

BASE = "https://api.coingecko.com/api/v3"

def extrair_mercado(vs="usd", total=250, por_pagina=250, tentativas=3):
    """Extrai o snapshot de mercado das top N moedas, paginando."""
    todas, pagina = [], 1
    while len(todas) < total:
        params = {"vs_currency": vs, "order": "market_cap_desc",
                  "per_page": por_pagina, "page": pagina}

        for t in range(tentativas):
            r = requests.get(f"{BASE}/coins/markets", params=params, timeout=30)
            if r.status_code == 429:                 # rate limit
                time.sleep(2 ** t)                   # backoff exponencial
                continue
            r.raise_for_status()
            lote = r.json()
            break
        else:
            # só chega aqui se o for terminou SEM break → todas as tentativas foram 429
            raise RuntimeError(
                f"Rate limit persistente na página {pagina} após {tentativas} tentativas"
            )

        if not lote:
            print("Não existem mais dados para coletar. Parando.")
            break
        todas.extend(lote)
        pagina += 1
    return todas[:total]

extracao = extrair_mercado()

for moeda in extracao[:5]:
    print(moeda['id'])

bitcoin
ethereum
tether
binancecoin
usd-coin


In [21]:
extracao

[{'id': 'bitcoin',
  'symbol': 'btc',
  'name': 'Bitcoin',
  'image': 'https://coin-images.coingecko.com/coins/images/1/large/bitcoin.png?1696501400',
  'current_price': 64312,
  'market_cap': 1290167290970,
  'market_cap_rank': 1,
  'fully_diluted_valuation': 1290167290970,
  'total_volume': 15541892601,
  'high_24h': 64254,
  'low_24h': 63761,
  'price_change_24h': 218.95,
  'price_change_percentage_24h': 0.5,
  'market_cap_change_24h': 4421191127,
  'market_cap_change_percentage_24h': 0.34386,
  'circulating_supply': 20061178.0,
  'total_supply': 20061178.0,
  'max_supply': 21000000.0,
  'ath': 126080,
  'ath_change_percentage': -48.99144,
  'ath_date': '2025-10-06T10:57:42.000Z',
  'atl': 67.81,
  'atl_change_percentage': 94742.17674,
  'atl_date': '2013-07-05T16:00:00.000Z',
  'roi': None,
  'last_updated': '2026-07-25T18:08:30.000Z'},
 {'id': 'ethereum',
  'symbol': 'eth',
  'name': 'Ethereum',
  'image': 'https://coin-images.coingecko.com/coins/images/279/large/ethereum.png?1696

In [14]:
import psycopg2
from datetime import datetime, timezone
# from extrair import extrair_mercado

In [15]:
# Transformar os dados

def transformar(bruto):
    """Seleciona e limpa apenas os campos que interessam."""
    agora = datetime.now(timezone.utc)
    return [(
        m["id"], m["symbol"], m["name"],
        m["current_price"], m["total_volume"],
        m["market_cap"], m.get("price_change_percentage_24h"), agora,
    ) for m in bruto]


dados_transformados = transformar(extracao)
dados_transformados

[]

In [16]:
# Importando dados para o banco

def carregar(linhas):
    """Grava no Postgres com UPSERT (idempotente na mesma coleta)."""
    conn = psycopg2.connect(
        host="localhost", port=5433, dbname="criptoflow",
        user="criptoflow", password="criptoflow",
    )
    with conn, conn.cursor() as cur:
        cur.executemany("""
            INSERT INTO mercado_bruto
            (id, simbolo, nome, preco_usd, volume_24h,
             market_cap, variacao_24h, coletado_em)
            VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
            ON CONFLICT (id, coletado_em) DO NOTHING
        """, linhas)
    conn.close()


carregar(dados_transformados)

In [25]:

## Carregar Moedas

def carregar_moedas(bruto):
    linhas = [(m['id'], m['symbol'], m['name']) for m in bruto]

    conn = psycopg2.connect(
        host="localhost", port=5433, dbname="criptoflow",
        user="criptoflow", password="criptoflow",
    )

    with conn, conn.cursor() as cur:
            cur.executemany("""
                INSERT INTO moedas
                (id, simbolo, nome)
                VALUES (%s,%s,%s)
                ON CONFLICT (id) DO NOTHING
            """, linhas)
    conn.close()


carregar_moedas(extracao)



In [ ]:
linhas = [(m['id'], m['symbol'], m['name']) for m in extracao]
linhas

conn = psycopg2.connect(
        host="localhost", port=5433, dbname="criptoflow",
        user="criptoflow", password="criptoflow",
    )

with conn, conn.cursor() as cur:
        cur.executemany("""
            INSERT INTO mercado_bruto
            (id, simbolo, nome, preco_usd, volume_24h,
             market_cap, variacao_24h, coletado_em)
            VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
            ON CONFLICT (id, coletado_em) DO NOTHING
        """, linhas)
    conn.close()

[('bitcoin', 'btc', 'Bitcoin'),
 ('ethereum', 'eth', 'Ethereum'),
 ('tether', 'usdt', 'Tether'),
 ('binancecoin', 'bnb', 'BNB'),
 ('usd-coin', 'usdc', 'USDC'),
 ('ripple', 'xrp', 'XRP'),
 ('solana', 'sol', 'Solana'),
 ('tron', 'trx', 'TRON'),
 ('figure-heloc', 'figr_heloc', 'Figure Heloc'),
 ('whitebit', 'wbt', 'WhiteBIT Coin'),
 ('hyperliquid', 'hype', 'Hyperliquid'),
 ('dogecoin', 'doge', 'Dogecoin'),
 ('usds', 'usds', 'USDS'),
 ('rain', 'rain', 'Rain'),
 ('leo-token', 'leo', 'LEO Token'),
 ('zcash', 'zec', 'Zcash'),
 ('monero', 'xmr', 'Monero'),
 ('chainlink', 'link', 'Chainlink'),
 ('cardano', 'ada', 'Cardano'),
 ('stellar', 'xlm', 'Stellar'),
 ('canton-network', 'cc', 'Canton'),
 ('dai', 'dai', 'Dai'),
 ('bitcoin-cash', 'bch', 'Bitcoin Cash'),
 ('usd1-wlfi', 'usd1', 'USD1'),
 ('the-open-network', 'gram', 'Gram (prev. Toncoin)'),
 ('ethena-usde', 'usde', 'Ethena USDe'),
 ('litecoin', 'ltc', 'Litecoin'),
 ('global-dollar', 'usdg', 'Global Dollar'),
 ('hedera-hashgraph', 'hbar', 'Hed